# PS07 - Autonomous Vision and Behaviour Understanding

**HackNEX 2026 | NEXUS Club, Karunya**

> **Other systems draw boxes. Ours writes incident reports, proves every claim, and catches accidents that almost happened.**

Give this notebook a video of a corridor, a shop floor, a road or a barn. It returns a short report that says
**who or what** (a stable ID) **did what** (near miss, fall, crowding, loitering, zone intrusion, running),
**when** (start and end time) and **why it counts** (the measured numbers, a confidence score, a severity and a
snapshot). It also cuts a **highlights.mp4** with only the incidents.

**How to use it**
1. `Runtime > Change runtime type > T4 GPU`, then save.
2. Run the cells from top to bottom (Shift+Enter). Cells marked *optional* can be skipped.
3. Section 5 chooses a **scenario** (campus, workplace, elderly, livestock, traffic) and the **privacy** switch.
4. Section 8 is the **near-miss demo**: the `traffic` preset on the Intel person-bicycle-car clip.
5. Fast loop for tuning thresholds: run section 7 once, then edit settings and use section 11 (`--reuse`).

Pipeline: `YOLO11n + ByteTrack -> per-object tracks -> rules -> merged events -> incident chains -> highlights + report`.

## 1. Check the GPU
You should see a Tesla T4 and `CUDA available: True`. Without a GPU everything still works, only slower
(then use `--device cpu` below).

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Get the code
Use **Option A or Option B, not both**, then run the `%cd` cell.

* **Option A (upload):** on your laptop zip the `ps07` folder **without** `.venv` and `outputs`, then upload `ps07.zip`.
* **Option B (git):** replace `REPO_URL` with your repository link and remove the `#`.

In [ ]:
# Option A: upload ps07.zip (a dialog opens)
import glob, os, shutil, zipfile
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
shutil.rmtree('/content/_unzip', ignore_errors=True)
with zipfile.ZipFile(zip_name) as zf:                 # extract by hand: Windows zips may use '\\' in paths
    for info in zf.infolist():
        name = info.filename.replace('\\', '/')
        target = os.path.join('/content/_unzip', name)
        if name.endswith('/'):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with zf.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)
found = glob.glob('/content/_unzip/**/run.py', recursive=True)      # works with or without a ps07/ folder inside
assert found, 'run.py not found inside the zip - did you zip the right folder?'
shutil.rmtree('/content/ps07', ignore_errors=True)
shutil.move(os.path.dirname(found[0]), '/content/ps07')
print('code is in /content/ps07')

In [ ]:
# Option B: clone from git (replace REPO_URL, then uncomment)
# !git clone REPO_URL /content/ps07

In [ ]:
%cd /content/ps07
!ls

## 3. Install the libraries
Colab already has PyTorch. This adds Ultralytics (YOLO11 + ByteTrack), the tracker's `lap` solver, and the rest.
If pip asks you to restart the runtime, do `Runtime > Restart session` and continue from the next cell.

In [ ]:
!pip install -q -r requirements.txt
import ultralytics
ultralytics.checks()

## 4. Get a video
The three Intel sample videos below are small and licensed CC BY 4.0 (credit: Intel Corporation).
* `one-by-one-person-detection.mp4`: people walking past a fixed camera (campus preset).
* `worker-zone-detection.mp4`: a good choice for trying a restricted zone.
* `person-bicycle-car-detection.mp4`: people, bicycles and cars in one scene (used by the near-miss demo in section 8).

To use **your own** video, uncomment the upload lines. The camera must be fixed (not moving).

In [ ]:
# Intel sample videos (CC BY 4.0) + the synthetic test clips (drawn locally, with ground truth)
!python get_samples.py
!ls -la samples

In [ ]:
import os

# Your own video instead (uncomment):
# from google.colab import files
# name = next(iter(files.upload()))
# !mv "{name}" samples/
# VIDEO = f'samples/{name}'

VIDEO = 'samples/one-by-one-person-detection.mp4'      # <- the video to analyse
STEM = os.path.splitext(os.path.basename(VIDEO))[0]
print('video:', VIDEO)

## 5. Choose a scenario and the privacy switch
A **scenario preset** changes what is tracked, which rules matter, the thresholds and the wording of the report:

| Preset | For | Headline |
|---|---|---|
| `campus` | corridor with a restricted zone (the default) | loitering, zone intrusion, running |
| `workplace` | industrial / workplace safety | near miss between a worker and a vehicle |
| `elderly` | elderly care | fall, no movement |
| `livestock` | animals in a pen | not moving, left the pen, stampede |
| `traffic` | roads and pedestrians | near miss between a pedestrian and a vehicle |

**Privacy mode** blurs faces in the annotated video, the highlight reel, the snapshots and the heatmap.
Detection and tracking still use the original frames, so the results do not change.

In [ ]:
!python run.py --list-scenarios

In [ ]:
# Pick a preset: campus | workplace | elderly | livestock | traffic | navigation   (the list above shows what each one does)
SCENARIO = 'campus'
PRIVACY = False            # True = blur faces in every output (video, highlights, snapshots, heatmap)

SCEN_ARG = '' if SCENARIO == 'campus' else f'--scenario {SCENARIO}'
PRIV_ARG = '--privacy' if PRIVACY else ''


def out_dir_for(video, scenario):
    # run.py writes to outputs/<video name>, plus _<preset> for every preset except campus
    stem = os.path.splitext(os.path.basename(video))[0]
    return f'outputs/{stem}' + ('' if scenario in ('', None, 'campus') else f'_{scenario}')


OUT = out_dir_for(VIDEO, SCENARIO)
print('scenario:', SCENARIO, '| privacy:', PRIVACY, '| results will be in', OUT)

## 6. Draw the restricted zone (optional)
Colab has no mouse-driven window, so we use two steps:
1. `--grid` writes `zone_grid.jpg`, the first frame with a coordinate grid (every 50 px).
2. Read the corners of the zone off the grid, put them in `POINTS` as `x,y` pairs, and run `--points`.

Coordinates are in **original video pixels**. The zone is a restricted area (campus), a machine danger zone
(workplace), the exit door (elderly), the pen (livestock) or a restricted lane (traffic).
Skip these cells to run without a zone.

In [ ]:
!python zone_picker.py --video "{VIDEO}" --grid
from IPython.display import Image, display
display(Image('zone_grid.jpg'))

In [ ]:
# Corners of the zone, in order around its edge. Replace with your own numbers.
POINTS = "400,200 760,200 760,430 400,430"
!python zone_picker.py --video "{VIDEO}" --points "{POINTS}" --out zones.json --name restricted
display(Image('zone_preview.jpg'))

## 7. Run the analysis
`--device 0` uses the T4 GPU (use `--device cpu` without a GPU). The first run downloads the YOLO weights once.
If you skipped the zone step, `ZONES` is empty and the zone and crowding rules have nothing to check; the others still run.
The console prints a table with a severity for every event and, below it, the **incident chains**
(several events of one entity linked into one story).

In [ ]:
ZONES = '--zones zones.json' if os.path.exists('zones.json') else ''
!python run.py --video "{VIDEO}" $ZONES $SCEN_ARG $PRIV_ARG --device 0

In [ ]:
# Variant WITHOUT a zone: separate output folder, no annotated video or highlights (faster).
!python run.py --video "{VIDEO}" $SCEN_ARG $PRIV_ARG --device 0 --out "outputs/{STEM}_nozones" --no-video

## 8. Near-miss demo: catch the accident that almost happened

An ordinary detector only reports accidents that happen. This system also reports the ones that **almost** happened:
two objects got closer than 0.6 body-heights while moving fast relative to each other (closing speed, time-to-collision).

Real near misses are rare in public footage, so the demo uses the **synthetic workplace clip** made by `get_samples.py`:
a cyclist nearly hits a pedestrian at about 6 s, and a worker falls at about 18 s. Its ground truth is in
`tests/synthetic_safety_labels.csv`, so the result can be scored (section 13).

The second command runs the **real** Intel car-park clip with the traffic preset. Nobody comes close to anyone there,
so the correct answer is **no incidents** (a true negative). Record your own clip (a bike or trolley passing close to a
walking person) to show a real near miss.

In [ ]:
DEMO_VIDEO = 'samples/synthetic_safety.mp4'
DEMO_OUT = 'outputs/synthetic_safety'
!python run.py --video "{DEMO_VIDEO}" --scenario workplace $PRIV_ARG --out "{DEMO_OUT}" --reuse

# Real footage, traffic preset (expected: no incidents - nobody comes close in this clip)
!python run.py --video samples/person-bicycle-car-detection.mp4 --scenario traffic $PRIV_ARG --device 0

## 9. Look at the results
In this order:
1. `highlights.mp4` is a short reel with only the incidents, captioned (title card, severity, fast-forward for long ones).
2. `report.html` has the incidents first, then one evidence card per event: snapshot, plot, times, confidence, reason.
3. `annotated.mp4` is the whole video with boxes, IDs, trails, zones and event labels.

`summary.txt` is the 3-6 line summary a guard can read.

In [ ]:
import base64, html as _html
from IPython.display import HTML, display


def show_video(path, width=720, max_mb=40):
    # Play an mp4 inside the notebook as a base64 data URL (very large files are skipped).
    if not os.path.exists(path):
        print('not written:', path)
        return
    size_mb = os.path.getsize(path) / 1e6
    print(f'{path}: {size_mb:.1f} MB')
    if size_mb < max_mb:
        b64 = base64.b64encode(open(path, 'rb').read()).decode()
        display(HTML(f'<video controls width="{width}" src="data:video/mp4;base64,{b64}"></video>'))
    else:
        print('Too big to embed - download it in the last section instead.')


def show_results(out):
    # summary, then 1) highlights.mp4, 2) report.html, 3) annotated.mp4
    print(open(f'{out}/summary.txt', encoding='utf-8').read())
    print('--- 1. highlights.mp4 (only the incidents) ---')
    show_video(f'{out}/highlights.mp4')
    print('--- 2. report.html (incidents and evidence cards) ---')
    page = open(f'{out}/report.html', encoding='utf-8').read()
    # The report is one self-contained HTML file; an iframe keeps its styling separate from the notebook.
    display(HTML(f'<iframe srcdoc="{_html.escape(page, quote=True)}" width="100%" height="900" '
                 f'style="border:1px solid #ccc;background:#fff"></iframe>'))
    print('--- 3. annotated.mp4 (the whole video) ---')
    show_video(f'{out}/annotated.mp4')


show_results(OUT)

In [ ]:
# The same for the near-miss demo of section 8:
show_results(DEMO_OUT)

## 10. Second opinion from body pose (optional)
`--pose` runs a pose model on a few frames of each running or loitering event and checks whether the body
agrees: running means leg spread or forward lean, loitering means an upright body. Each event gets
`verified = True`, `False` or `"uncertain"`. `--reuse` skips the detector, so this is quick.

In [ ]:
# Separate folder so the section 7 results stay intact; reuse its detections (no second YOLO pass).
POSE_OUT = OUT + '_pose'
!mkdir -p "{POSE_OUT}" && cp "{OUT}/detections.json" "{POSE_OUT}/"
!python run.py --video "{VIDEO}" $ZONES $SCEN_ARG $PRIV_ARG --device 0 --pose --reuse --no-video --out "{POSE_OUT}"
import json
for e in json.load(open(f'{POSE_OUT}/events.json'))['events']:
    note = (e.get('pose') or {}).get('note')
    who = e.get('entity_name') or f"#{e['entity_id']}"
    print(f"event {e['event_id']} {who} {e['behavior']:<15} verified = {e['verified']}  {note or ''}")

## 11. Tune the thresholds, rerun in seconds
All thresholds live in `config.yaml` (distances are in body-heights or body-lengths, so they work for near and far objects).
`--reuse` loads the saved `detections.json` and **skips YOLO**, so only the rules run again (about a second).
It is valid as long as `video.frame_stride`, `video.resize_width` and `model.*` (including the tracked classes) are unchanged.
If you change those, rerun without `--reuse`.

A scenario preset is merged **on top of** the config file you pass, so a value the preset sets (for example the running
speed in `traffic`) wins over your tuned file. Tune a value the preset does not set, or edit the preset itself
(`scenarios/<name>.yaml`, or copy it and pass `--scenario my_preset.yaml`).

In [ ]:
!sed -n '/^behaviors:/,/^events:/p' config.yaml

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['behaviors']['near_miss']['near_distance_bh'] = 0.8   # more sensitive: closer than 0.8 body-heights counts
cfg['behaviors']['near_miss']['ttc_s'] = 1.5               # ...or a collision predicted within 1.5 s
cfg['behaviors']['fall']['min_down_s'] = 3                 # a fall must last 3 s
yaml.safe_dump(cfg, open('config_tuned.yaml', 'w'), sort_keys=False)

TUNED_OUT = OUT + '_tuned'                                  # own folder: section 7 results stay intact
!mkdir -p "{TUNED_OUT}" && cp "{OUT}/detections.json" "{TUNED_OUT}/"
!python run.py --video "{VIDEO}" $ZONES $SCEN_ARG $PRIV_ARG --config config_tuned.yaml --reuse --no-video --out "{TUNED_OUT}"
print(open(f'{TUNED_OUT}/summary.txt').read())

## 12. Batch: every video in `samples/` (optional)
Writes one folder per video under `outputs/` (with the preset name added unless it is `campus`).
Zones are camera-specific, so none are used here. The scenario chosen in section 5 applies to all videos.

In [ ]:
import glob
for v in sorted(glob.glob('samples/*.mp4')):
    o = 'outputs/batch/' + os.path.splitext(os.path.basename(v))[0]   # one folder per clip, apart from section 7
    print('=====', v, '->', o)
    !python run.py --video "{v}" $SCEN_ARG $PRIV_ARG --device 0 --no-video --out "{o}"

## 13. Measure accuracy against your own labels (optional)
Watch a clip and write down the true incidents in `labels.csv`:
`clip,entity_description,behavior,start_s,end_s` (see `labels_template.csv`; `none` marks a clip with no incidents).
`behavior` can be `loitering`, `zone_intrusion`, `running`, `fall`, `crowding` or `near_miss`.
Upload your `labels.csv` or edit the copy below. Matching allows 2 s of slack, and the script prints
precision, recall, F1 and timing errors per behaviour.

In [ ]:
# Ready-made example: score the synthetic workplace clip of section 8 against its ground truth.
!python evaluate.py --labels tests/synthetic_safety_labels.csv --events outputs/synthetic_safety/events.json --clip synthetic_safety.mp4

# Your own clips: write labels.csv (one row per incident you see), then score every folder in outputs/.
if os.path.exists('labels.csv'):
    !python evaluate.py --labels labels.csv --outputs outputs/
else:
    !cp labels_template.csv labels.csv
    print('Copied the template to labels.csv - replace the example rows with your own incidents, then rerun.')

## 14. Download everything
Zips the whole `outputs/` folder (reports, highlight reels, snapshots, plots, annotated videos, `events.json`) and downloads it.

In [ ]:
import shutil
from google.colab import files

for extra in ['zones.json', 'config_tuned.yaml', 'eval_report.txt', 'eval_report.json']:
    if os.path.exists(extra):
        shutil.copy(extra, 'outputs/')
shutil.make_archive('ps07_outputs', 'zip', 'outputs')
files.download('ps07_outputs.zip')

## 15. Train the swimmer detector (pool scenario, about 30-40 min on a T4)

Downloads the public pool datasets (Roboflow Universe, CC BY 4.0), merges them into our 5 body-state classes
(duplicates removed, split by video so the test score is honest), fine-tunes YOLO11n, evaluates it on the
test split and gives you `best.pt` to plug into `scenarios/pool.yaml` (see training/README.md).

**Your Roboflow key stays private:** click the **key icon (Secrets)** in Colab's left sidebar, add a secret named
`ROBOFLOW_API_KEY` with your *Private API Key*, and allow this notebook to access it. Never paste the key into a cell.

In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')   # read from Colab Secrets, never printed
!python training/get_datasets.py
!python training/build_dataset.py

In [ ]:
# Full training on the T4 (lower --epochs if you are short of time; 30 epochs ~ 20 min)
!python training/train_swimmer.py --data datasets/swimmers/data.yaml --epochs 60 --device 0

In [ ]:
# Look at the training curves and the confusion matrix, then download the model
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png']:
    p = f'training/runs/swimmer/{f}'
    if os.path.exists(p):
        display(Image(p, width=900))
from google.colab import files
files.download('training/runs/swimmer/weights/best.pt')
files.download('training/runs/swimmer/eval.json')